# SEIS 606: Vibe Coding
## Homework 2, Image Generation for App Specs
Dante Razo, razo3843@stthomas.edu, FA26

I've been using this GPU-accelerated notebook template since I first started at UST. It's something that I carry from class to class.

## GPU-Accelerated Environment Configuration

In [ ]:
import torch

# validate CUDA setup
print("Torch CUDA Available? ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Torch CUDA Version:", torch.version.cuda)
    print("Torch cuDNN Version:", torch.backends.cudnn.version())

    # print GPU information
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(device=i))

    # check NVIDIA driver
    !echo && nvidia-smi

# set device type
device: str = "cuda" if torch.cuda.is_available() else "cpu"

Torch CUDA Available?  True
Torch CUDA Version: 13.0
Torch cuDNN Version: 92400

GPU 0: NVIDIA GeForce RTX 5090

Fri Sep 25 01:03:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 615.71.08              KMD Version: 616.92        CUDA UMD Version: 13.4     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:0A:00.0  On |                  N/A |
|  0%   44C    P0             64W /  460W |    3563MiB /  32607MiB |      4%      Default |
|                          

In [2]:
import gc


def free_vram() -> None:
    gc.collect()
    torch.cuda.empty_cache()


# free now, though it should be empty with a fresh kernel
free_vram()

In [3]:
import os
from pathlib import Path

# create cache location
hf_home: Path = Path("/cache/huggingface")
hf_home.mkdir(parents=True, exist_ok=True)

# set environment variables for huggingface / transformers
os.environ["HF_HOME"] = str(object=hf_home)

In [4]:
from dotenv import load_dotenv

# load environment, including HF token
load_dotenv()

False

In [5]:
# validate environment variables with assertions
assert hf_home.exists()
assert os.environ["HF_HOME"] == str(object=hf_home)

## Loading the Image Generation Model
For simplicity and compatibility with a 32GB RTX 5090, this notebook uses **SDXL 1.0** in native fp16 weights from Hugging Face (no runtime quantization).

In [ ]:
from diffusers.pipelines.auto_pipeline import AutoPipelineForText2Image

# load model into VRAM
pipeline = AutoPipelineForText2Image.from_pretrained(
    pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True,
).to(device)

# improve memory efficiency
pipeline.enable_attention_slicing()

/home/dante/code/coursework/seis-606-vibe/.venv/lib/python3.14/site-packages/diffusers/quantizers/quantization_config.py:671: FutureWarning: `QuantoConfig` is deprecated and will be removed in version 1.0.0. `QuantoConfig` is deprecated and will be removed in version 1.0.0.
  deprecate("QuantoConfig", "1.0.0", deprecation_message)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
# wrapper function for generation + persisting to disk
def generate_image(prompt: str, save_path: str = "") -> None:
    with torch.inference_mode():
        image = pipeline(
            prompt=prompt,
            num_inference_steps=30,
            guidance_scale=7.0,
        ).images[0]
    image.save(save_path) if save_path else None
    display(image)

In [ ]:
generate_image(prompt="A clean UI mockup for a homelab dashboard")